In [0]:
dbutils.widgets.text("storage_account_name", "portfolioalba")
dbutils.widgets.text("container_name", "raw-data")


storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

access_key = dbutils.secrets.get(scope="portfolio-secrets", key="datalake-key")

spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", access_key)

file_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/clean_train.csv"

df_spark = spark.read.csv(file_path, header=True, inferSchema=True)

df_spark.show(5)


+--------------------+------------------+----------------+
|                text|             brand|       sentiment|
+--------------------+------------------+----------------+
|i have a 3g iphon...|            iPhone|Negative emotion|
|know about   awes...|iPad or iPhone App|Positive emotion|
|can not wait for ...|              iPad|Positive emotion|
|i hope this years...|iPad or iPhone App|Negative emotion|
|great stuff on fr...|            Google|Positive emotion|
+--------------------+------------------+----------------+
only showing top 5 rows


### Group by the “sentiment” column and count how many rows there are for each

In [0]:
from pyspark.sql.functions import count

record_df = df_spark.groupBy("sentiment").agg(count("*").alias("total_tweets"))

record_df.show()

+--------------------+------------+
|           sentiment|total_tweets|
+--------------------+------------+
|    Negative emotion|         519|
|                NULL|         164|
|No emotion toward...|        5388|
|    Positive emotion|        2672|
+--------------------+------------+



### Saving the DataFrame in Parquet format

In [0]:
output_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/processed_data/sentiment_record"

record_df.write.mode("overwrite").parquet(output_path)

print(f"Data written to {output_path}")
                                          

Data written to abfss://raw-data@portfolioalba.dfs.core.windows.net/processed_data/sentiment_record


### Sending data to Azure SQL Database

In [0]:
dbutils.widgets.text("server_name", "sentiments-server.database.windows.net")
dbutils.widgets.text("database_name", "sentiments-db")

server_name = dbutils.widgets.get("server_name")
database_name = dbutils.widgets.get("database_name")
username = "CloudSAb35b1b88"
password = dbutils.secrets.get(scope="portfolio-secrets", key="sql-password")

jdbc_url = f"jdbc:sqlserver://{server_name}:1433;database={database_name}"

connection_properties = {
    "user": username,
    "password": password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

record_df.write.jdbc(
    url=jdbc_url, 
    table="SentimentsRecord", 
    mode="overwrite", 
    properties=connection_properties
)

print("Data sent to Azure SQL Database successfully")


Data sent to Azure SQL Database successfully
